# Exploración del Titanic

Análisis exploratorio del taller de Buenas Prácticas en Ciencia de Datos e IA.

Este cuaderno lee la capa **Silver** que produce `flows/01_data_engineering.py`,
nunca la fuente original. Para reproducirlo desde cero:

In [ ]:
# 1) Importamos las librerías con sus alias convencionales
import pandas as pd                 # tablas (DataFrame)
import numpy as np                  # cálculo numérico y condiciones vectorizadas
import matplotlib.pyplot as plt     # gráficos base (Seaborn se apoya en él)
import seaborn as sns               # gráficos estadísticos de alto nivel
from scipy import stats             # pruebas estadísticas (Parte 6)

from flows.common.paths import BRONZE, SILVER, GOLD, BRONZE_LABELED_NAME, SILVER_NAME

# 2) Estilo global: todos los gráficos heredan esta configuración
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)   # tamaño por defecto (ancho, alto) en pulgadas
plt.rcParams['axes.titlesize'] = 13

# 3) A partir de la Parte 2 el cuaderno lee Silver, no Bronze:
#    consume lo que produce el pipeline, no el CSV crudo.
df = pd.read_parquet(SILVER / SILVER_NAME)

## Parte 1 — Primer contacto con los datos

### Pasos 1 a 4. Forma, tipos y estadísticos

Cuántas filas y columnas hay, de qué tipo es cada una, y cómo se distribuyen los valores numéricos y los categóricos.

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(exclude='number').describe()

### Paso 5. Datos faltantes

Dónde faltan datos, cuánto falta en cada columna, y si los huecos siguen un patrón o están repartidos al azar.

In [ ]:
# Conteo de valores faltantes por columna
df.isnull().sum()

In [ ]:
# Solo las columnas incompletas, con su porcentaje, de mayor a menor
faltantes = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "pct": (df.isnull().sum() / len(df) * 100).round(2),
})
faltantes[faltantes["nulos"] > 0].sort_values("nulos", ascending=False)

In [ ]:
# Mapa de datos faltantes: cada franja clara es un valor ausente.
# Permite ver si los huecos siguen un patrón o están repartidos al azar.
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Datos faltantes en Silver (891 pasajeros)')
plt.show()

In [ ]:
# ¿Falta Cabin por igual en todas las clases? Proporción de faltantes por clase
pd.crosstab(df['Pclass'], df['Cabin'].isnull(), normalize='index').round(3)

**Lectura del cruce.** Los faltantes de `Cabin` no son aleatorios: falta en el 18.5 % de primera clase y en el 97.6 % de tercera.

El dato ausente **no significa que el pasajero no tuviera camarote**: significa que no quedó registrado. Lo más probable es que refleje cómo se compilaron los manifiestos, con más detalle para primera clase, y no una diferencia física en el alojamiento. Es una asociación, no una causa.

Consecuencia para el modelado: `Cabin` no se imputa. Y una variable binaria "tiene camarote" sería casi un duplicado de `Pclass`.

## Parte 2 — Conociendo a los pasajeros

### Paso 6. Distribución por género y su cruce con la clase

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por género')
plt.show()

In [ ]:
sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Género por clase')
plt.show()

In [ ]:
# Normalizado POR COLUMNAS: cada clase suma 1, así se comparan
# clases de tamaño distinto sin que la más grande domine.
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns').round(3)

### Paso 7. Distribución de edades

`Age` todavía **no está imputada** en Silver (eso llega en el Paso 22), así que aquí solo se ven los 714 pasajeros con edad conocida.

In [ ]:
sns.histplot(df['Age'].dropna(), bins=30)
plt.title('Distribución de edades (solo pasajeros con edad conocida)')
plt.xlabel('Edad')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# Pandas ignora los NaN automáticamente en estas agregaciones
df['Age'].agg(['mean', 'median', 'min', 'max']).round(2)

### Paso 8. La variable `Person`

`Person` la crea `add_person()` en `flows/01_data_engineering.py`, no el cuaderno:
`'child'` si `Age < 16`, y el valor de `Sex` en caso contrario.

**Regla explícita para edad desconocida:** en NumPy, `NaN < 16` es `False`, así que un pasajero sin edad nunca se marca como `child`. Eso subestima a los niños sin edad registrada, y el Reto 8.1 cuantifica cuánto.

In [ ]:
df['Person'].value_counts()

In [ ]:
# Reto 8.1: ¿cuántos 'child' se pierden por falta de edad registrada?
sin_edad = int(df['Age'].isna().sum())
p_ninos = (df.loc[df['Age'].notna(), 'Age'] < 16).mean()
print(f"Pasajeros sin edad registrada: {sin_edad}")
print(f"Menores de 16 entre los que sí tienen edad: {p_ninos:.1%}")
print(f"Estimación de 'child' no contados: ~{round(sin_edad * p_ninos)}")

### Paso 9. ¿Cuántos niños viajaban en cada clase?

In [ ]:
pd.crosstab(df['Pclass'], df['Person'])

### Paso 10. Densidades de edad (KDE)

Un KDE es un histograma suavizado: la curva muestra dónde se concentran los
valores. `common_norm=False` normaliza **cada grupo por separado**; sin eso, los
grupos grandes aplastan visualmente a los pequeños y el gráfico engaña.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

sns.kdeplot(data=df, x='Age', hue='Sex', fill=True, common_norm=False, ax=axes[0])
axes[0].set_title('Edad por sexo')

sns.kdeplot(data=df, x='Age', hue='Person', fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('Edad por tipo de pasajero')

sns.kdeplot(data=df, x='Age', hue='Pclass', fill=True, common_norm=False,
            palette='viridis', ax=axes[2])
axes[2].set_title('Edad por clase')

plt.tight_layout()
plt.show()

## Parte 3 — Ubicación, embarque y composición familiar

### Paso 11. La cubierta del barco

`add_cabin_features()` extrae dos columnas de `Cabin`: `TCabin` con la letra de
la cubierta (`'C85'` → `'C'`, y `'X'` cuando no hay cabina registrada), y
`HasCabin` con 0/1.

Separarlas importa porque el Paso 5 mostró que **la ausencia del dato no es
aleatoria**: `HasCabin` captura ese hecho, `TCabin` captura el contenido.

In [ ]:
df['TCabin'].value_counts().sort_index()

### Paso 12. Conteo por cubierta, con depuración de anómalos

In [ ]:
# Solo las cabinas conocidas: la 'X' no es una cubierta, es la ausencia del dato
cubierta = df.loc[df['TCabin'] != 'X', 'TCabin']

sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros por cubierta (solo cabinas registradas)')
plt.xlabel('Cubierta')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# ¿Hay alguna cubierta con un solo registro? Se detecta, no se asume.
conteo = cubierta.value_counts()
print(conteo.sort_index())
print(f"\nCubiertas con un solo pasajero: {conteo[conteo == 1].index.tolist()}")

Una cubierta con un solo pasajero **no es un dato erróneo**: ese pasajero
existió y su cabina está registrada. Es un problema de estadística, no de
calidad del dato — un grupo de tamaño 1 no permite estimar nada y distorsiona
la escala del gráfico.

Por eso se excluye **solo del gráfico** y la fila se conserva intacta en Silver.

In [ ]:
cubierta_sin_anomala = cubierta[cubierta != 'T']

sns.countplot(x=cubierta_sin_anomala, order=sorted(cubierta_sin_anomala.unique()))
plt.title('Pasajeros por cubierta (sin la cubierta T, de un solo registro)')
plt.xlabel('Cubierta')
plt.ylabel('Pasajeros')
plt.show()

### Paso 13. Puerto de embarque y clase

`S` = Southampton, `C` = Cherburgo, `Q` = Queenstown.

In [ ]:
sns.countplot(data=df, x='Embarked', hue='Pclass')
plt.title('Puerto de embarque por clase')
plt.xlabel('Puerto')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# Normalizado por FILAS: cada puerto suma 1.
# Responde "¿qué composición de clases tenía cada puerto?",
# no "¿de dónde venía cada clase?" — son preguntas distintas.
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(3)

### Paso 14. Viajar solo o acompañado

`add_family_features()` calcula `Familia = SibSp + Parch` (familiares a bordo,
sin contar al pasajero) y `Alone` como su versión categórica.

In [ ]:
df['Alone'].value_counts(normalize=True).round(3)

In [ ]:
# El tamaño del grupo, que 'Alone' pierde al reducir todo a dos categorías
df['Familia'].value_counts().sort_index()

### Paso 15. Distribución de `Alone`

In [ ]:
sns.countplot(data=df, x='Alone')
plt.title('Pasajeros que viajaban solos frente a los que iban con familia')
plt.xlabel('')
plt.ylabel('Pasajeros')
plt.show()

## Parte 4 — Análisis de supervivencia

La media de una variable 0/1 **es una proporción**: `df['Survived'].mean()` da
directamente la tasa de supervivencia. Por eso los gráficos que promedian usan
`Survived` (numérica) y no `Survivor` (texto), que solo sirve para etiquetar.

Las **barras verticales** de los gráficos de puntos son intervalos de confianza
al 95 %: miden incertidumbre. Barras largas significan pocos datos o mucha
variabilidad, y conviene desconfiar de esas diferencias.

### Paso 16. Panorama general

In [ ]:
sns.countplot(data=df, x='Survivor', order=['No', 'Yes'])
plt.title('¿Sobrevivió?')
plt.xlabel('')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# La media de una columna 0/1 es la proporción de unos
tasa = df['Survived'].mean()
print(f"Tasa general de supervivencia: {tasa:.4f}  ({tasa:.1%})")
print(f"Sobrevivieron {df['Survived'].sum()} de {len(df)} pasajeros")

### Paso 17. Supervivencia por clase

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.4)
plt.title('Tasa de supervivencia por clase')
plt.xlabel('Clase')
plt.ylabel('Tasa de supervivencia')
plt.show()

### Paso 18. Supervivencia por clase y tipo de pasajero

Aquí se cruzan las dos reglas en tensión: *"mujeres y niños primero"* contra el
acceso desigual a los botes según la clase. El Paso 9 mostró que 58 de los 83
niños del barco viajaban en tercera.

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', hue='Person',
            hue_order=['child', 'female', 'male'],
            kind='point', height=4, aspect=1.4)
plt.title('Supervivencia por clase y tipo de pasajero')
plt.xlabel('Clase')
plt.ylabel('Tasa de supervivencia')
plt.show()

Los valores del gráfico se leen a ojo. La tabla da las cifras exactas y, sobre
todo, **cuántos pasajeros hay detrás de cada una**: una tasa del 100 % con 19
casos dice algo; con 6, mucho menos.

In [ ]:
# Tasas exactas y, crucialmente, cuántos pasajeros hay detrás de cada una
pd.concat({
    'tasa': df.pivot_table(values='Survived', index='Pclass', columns='Person', aggfunc='mean').round(3),
    'n':    df.pivot_table(values='Survived', index='Pclass', columns='Person', aggfunc='count'),
}, axis=1)

### Paso 19. Edad y supervivencia

`lmplot` ajusta una recta de regresión. Con `y` binaria la recta es solo una
**aproximación visual** de la tendencia, no un modelo válido. `x_bins` agrupa la
edad en intervalos y dibuja la tasa media de cada uno, que es mucho más honesto.

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived')
plt.title('Edad y supervivencia (recta de regresión)')
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass')
plt.title('Edad y supervivencia, por clase')
plt.show()

In [ ]:
# x_bins agrupa la edad y muestra la tasa media de cada intervalo:
# más legible y más honesto que la recta
generaciones = [10, 20, 40, 60, 80]

sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass', x_bins=generaciones)
plt.title('Tasa de supervivencia por generación y clase')
plt.ylabel('Tasa de supervivencia')
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived', hue='Sex', x_bins=generaciones)
plt.title('Tasa de supervivencia por generación y sexo')
plt.ylabel('Tasa de supervivencia')
plt.show()

### Paso 20. Relación entre tarifa y edad

In [ ]:
plt.scatter(df['Age'], df['Fare'], alpha=0.5)
plt.title('Tarifa frente a edad')
plt.xlabel('Edad')
plt.ylabel('Tarifa')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Age', y='Fare', hue='Pclass', alpha=0.5, palette='viridis')
plt.title('Tarifa frente a edad, por clase')
plt.xlabel('Edad')
plt.ylabel('Tarifa')
plt.show()

## Parte 5 — Limpieza y preparación

Un dato faltante puede **eliminarse**, **imputarse** o **conservarse como
categoría propia**. Ninguna opción es neutral: cada una cambia lo que los datos
dicen. Esta parte documenta qué se hizo y por qué.

### Paso 21. `Embarked`

`impute_embarked()` rellena los 2 faltantes con la moda. Silver ya llega
imputada, así que aquí se verifica el resultado.

In [ ]:
print(f"Nulos en Embarked: {df['Embarked'].isna().sum()}")
df['Embarked'].value_counts()

In [ ]:
# La codificación fija: S -> 0, C -> 1, Q -> 2
pd.crosstab(df['Embarked'], df['NEmbarked'])

### Paso 22. `Age` y el Reto 22.1

`impute_age()` rellena con el promedio por género y conserva la edad real en
`AgeOriginal`. Comparar las dos columnas muestra qué le hace una imputación a
la forma de una distribución.

In [ ]:
print(f"Edades reales:    {df['AgeOriginal'].notna().sum()}")
print(f"Edades imputadas: {df['AgeImputada'].sum()}")
print()
print(f"Desviación estándar ANTES:   {df['AgeOriginal'].std():.2f}")
print(f"Desviación estándar DESPUÉS: {df['Age'].std():.2f}")
print()
print("Valores imputados (promedio de cada grupo):")
print(df.loc[df['AgeImputada'], ['Sex', 'Age']].drop_duplicates().to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

sns.histplot(df['AgeOriginal'].dropna(), bins=30, ax=axes[0])
axes[0].set_title('Age original (solo las 714 conocidas)')
axes[0].set_xlabel('Edad')

sns.histplot(df['Age'], bins=30, ax=axes[1])
axes[1].set_title('Age tras imputar (891)')
axes[1].set_xlabel('Edad')

plt.tight_layout()
plt.show()

### Pasos 23 y 24. Codificaciones de `Sex` y `Cabin`

In [ ]:
pd.crosstab(df['Sex'], df['NSex'])

In [ ]:
# NCabin usa el mapa FIJO de DECKS (A=0 ... T=7, X=8), no el orden de los datos
pd.crosstab(df['TCabin'], df['NCabin'])

### Paso 25. Títulos extraídos del nombre

Silver ya trae `TTitulo` **consolidado**, así que los títulos originales no
están. Se vuelven a extraer aquí solo para explorar y ver por qué hacía falta
consolidarlos.

In [ ]:
# Se re-extrae solo para explorar: así se ven los títulos raros que Silver ya agrupó
titulos_crudos = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
titulos_crudos.value_counts()

In [ ]:
pd.crosstab(titulos_crudos, df['Sex'])

### Paso 26. Títulos consolidados

`TITLE_MAP` funde los equivalentes (`Mme` → `Mrs`, `Mlle` y `Ms` → `Miss`,
`Master` → `Mr`) y todo lo que no esté en `MAIN_TITLES` pasa a `'Otro'`.

In [ ]:
df['TTitulo'].value_counts()

In [ ]:
# ¿Cómo quedó cada título crudo tras consolidar?
pd.crosstab(titulos_crudos, df['TTitulo'])

## Parte 6 — Correlaciones y estadísticas por grupos

### Paso 27. Matriz de correlación

La correlación de Pearson mide relación **lineal** entre pares de variables,
de −1 a +1. Tres límites que conviene tener presentes: no implica causalidad,
no detecta relaciones no lineales, y es poco informativa sobre categóricas
codificadas con números arbitrarios (`NCabin`, `NEmbarked`, `NTitulo`).

In [ ]:
cols = ['Survived', 'Pclass', 'NSex', 'Age', 'SibSp', 'Parch',
        'Fare', 'NEmbarked', 'NCabin', 'NTitulo']
corr = df[cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación')
plt.show()

In [ ]:
# Solo la columna que importa, ordenada de la correlación más negativa a la más positiva
corr['Survived'].sort_values()

### Paso 28. Estadísticas por grupos (capa Gold)

Estas tablas **no se calculan aquí**: las produce `flows/03_aggregation.py` y
quedan guardadas en `data/gold/`. El cuaderno solo las consume, igual que
consume Silver en lugar del CSV crudo.

Cada tabla trae `mean`, `sum` y `count` juntos: la tasa sola engaña si no se
sabe cuántos pasajeros hay detrás.

In [ ]:
pd.read_parquet(GOLD / 'survival_by_sex.parquet').round(3)

In [ ]:
pd.read_parquet(GOLD / 'survival_by_pclass.parquet').round(3)

In [ ]:
pd.read_parquet(GOLD / 'survival_by_alone.parquet').round(3)

In [ ]:
pd.read_parquet(GOLD / 'age_by_survival.parquet').round(2)

### Paso 29. ¿La diferencia por sexo es estadísticamente significativa?

Prueba de independencia **chi-cuadrado**. Compara los conteos observados con
los que se esperarían si las dos variables fueran independientes.

- **H₀:** `Sex` y `Survived` son independientes.
- **H₁:** hay asociación.

Con `p < 0.05` se rechaza H₀.

In [ ]:
import json

prueba = json.loads((GOLD / 'chi2_sex_survival.json').read_text(encoding='utf-8'))
print(f"chi2    = {prueba['chi2']:.2f}")
print(f"p-valor = {prueba['p_value']:.3g}")
print(f"gl      = {prueba['dof']}")
print()
print("Tabla observada:")
print(pd.crosstab(df['Sex'], df['Survived']).to_string())

**Conclusión, con sus límites.**

Con p ≈ 1.2 × 10⁻⁵⁸ se rechaza H₀: la asociación entre sexo y supervivencia no
es atribuible al azar.

Pero el p-valor **no dice** tres cosas, y conviene ser explícito:

1. **No dice que el efecto sea grande.** El tamaño está en las tasas —74.2 %
   frente a 18.9 %—, no en el chi-cuadrado. Con muestras grandes, diferencias
   triviales también dan p pequeños.
2. **No dice que sea causal.** Que ser mujer se asocie con sobrevivir no prueba
   que el sexo fuera la causa; podría mediar el acceso a los botes, la clase, o
   el protocolo de evacuación.
3. **No controla por otras variables.** El Paso 18 ya mostró que el efecto
   cambia mucho según la clase: las mujeres de tercera sobrevivieron al 49 %
   frente al 98 % de las de primera. Esta prueba trata a todas por igual.

La Parte 9 retoma esto estratificando, que es la forma correcta de separar
efectos confundidos.

## Parte 7 — Visualizaciones comparativas avanzadas

### Paso 30. Boxplots por grupo

La caja va del cuartil 1 al 3 (el 50 % central), la línea interior es la
mediana, los bigotes llegan hasta 1.5 × IQR y los puntos sueltos son
**candidatos a atípicos**.

Se usa **`AgeOriginal`**, no `Age`: la imputación concentró 177 pasajeros en
solo dos valores, lo que desplaza medianas y cuartiles hacia el centro. Con la
edad real se ven las distribuciones que los datos tienen de verdad.

In [ ]:
import warnings

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# seaborn 0.13 pasa a matplotlib 3.11 un parámetro ya deprecado. Es ruido de
# versiones, no un problema del análisis; se silencia SOLO en este bloque para
# no ocultar advertencias legítimas en el resto del cuaderno.
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='vert: bool was deprecated')

    sns.boxplot(data=df, x='Sex', y='AgeOriginal', ax=axes[0, 0])
    axes[0, 0].set_title('Edad por sexo')

    sns.boxplot(data=df, x='Pclass', y='AgeOriginal', ax=axes[0, 1])
    axes[0, 1].set_title('Edad por clase')

    sns.boxplot(data=df, x='Embarked', y='AgeOriginal', ax=axes[1, 0])
    axes[1, 0].set_title('Edad por puerto de embarque')

    sns.boxplot(data=df, x='Survived', y='AgeOriginal', ax=axes[1, 1])
    axes[1, 1].set_title('Edad por supervivencia')

plt.tight_layout()
plt.show()

**La pregunta del Paso 30: ¿`Age` o `AgeOriginal`?**

La comparación siguiente muestra por qué importa la elección.

In [ ]:
# La misma variable, antes y después de imputar: la imputación aplana la caja
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='vert: bool was deprecated')
    sns.boxplot(data=df, x='Sex', y='AgeOriginal', ax=axes[0])
    axes[0].set_title('AgeOriginal (714 edades reales)')
    sns.boxplot(data=df, x='Sex', y='Age', ax=axes[1])
    axes[1].set_title('Age imputada (891)')

plt.tight_layout()
plt.show()

print("Rango intercuartílico (IQR) por sexo:")
for col in ('AgeOriginal', 'Age'):
    iqr = df.groupby('Sex')[col].quantile(0.75) - df.groupby('Sex')[col].quantile(0.25)
    print(f"  {col:12s} female {iqr['female']:5.2f}   male {iqr['male']:5.2f}")

### Paso 31. Distribución de la tarifa

In [ ]:
sns.histplot(df['Fare'], kde=True, bins=40)
plt.title('Distribución de la tarifa')
plt.xlabel('Tarifa')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
print(f"Media:     {df['Fare'].mean():.2f}")
print(f"Mediana:   {df['Fare'].median():.2f}")
print(f"Asimetría: {df['Fare'].skew():.2f}")
print()
print(f"La media es {df['Fare'].mean() / df['Fare'].median():.2f} veces la mediana")
print(f"El 1 % más caro empieza en: {df['Fare'].quantile(0.99):.2f}")
print(f"Máximo: {df['Fare'].max():.2f}")

## Parte 8 — Dataset final y verificación

### Paso 32. Las dos vistas de Silver

`drop_original_columns()` produce una **copia** sin `Name`, `Sex`, `Embarked` ni
`Cabin`. El pipeline guarda las dos:

- **`titanic_silver.parquet`** conserva todo, para el análisis crítico de la
  Parte 9. Nunca se descarta información en silencio.
- **`titanic_final.parquet`** es la vista reducida que alimentará el modelado.

In [ ]:
final = pd.read_parquet(SILVER / 'titanic_final.parquet')

print(f"Silver completo: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Vista final:     {final.shape[0]} filas x {final.shape[1]} columnas")
print()
print(f"Columnas que quedaron fuera: {sorted(set(df.columns) - set(final.columns))}")

In [ ]:
# Cada columna eliminada tiene su reemplazo derivado: no se perdió información
reemplazos = {'Sex': 'NSex', 'Embarked': 'NEmbarked',
              'Cabin': 'TCabin / HasCabin / NCabin', 'Name': 'TTitulo / NTitulo'}
for original, derivadas in reemplazos.items():
    print(f"  {original:10s} -> {derivadas}")

### Paso 33. Los contratos de entrada y de salida

Una capa es confiable solo si **falla ruidosamente** cuando algo no cumple su
contrato. `validate_bronze()` vigila la entrada y `validate_silver()` la salida.

Las celdas siguientes los ponen a prueba a propósito, con datos rotos.

In [ ]:
import importlib
de = importlib.import_module('flows.01_data_engineering')

# 1) Bronze sin una columna obligatoria
try:
    de.validate_bronze(df.drop(columns=['Fare']))
    print("[contrato de entrada] NO falló  <-- problema")
except ValueError as e:
    print(f"[contrato de entrada] ValueError: {e}")

# 2) Silver con un nulo en una columna derivada
roto = df.copy()
roto.loc[0, 'NSex'] = None
try:
    de.validate_silver(roto)
    print("[contrato de salida]  NO falló  <-- problema")
except ValueError as e:
    print(f"[contrato de salida]  ValueError: {e}")

# 3) Silver intacta: debe pasar sin ruido
de.validate_silver(df)
print("[Silver real]         pasa el contrato")

In [ ]:
# AgeOriginal queda FUERA del contrato a propósito: sus nulos son el registro
# de qué edades no se conocían, y perderlos haría imposible auditar la imputación
print(f"Nulos en AgeOriginal: {df['AgeOriginal'].isna().sum()}  (el contrato los permite)")
print(f"Nulos en Age:         {df['Age'].isna().sum()}  (el contrato los prohíbe)")

## Parte 9 — Análisis crítico

Aquí un número solo no basta. Hay que preguntarse si es **grande**, si es
**confiable** (¿cuántos pasajeros hay detrás?) y si es **justo compararlo**
(¿hay una tercera variable que confunde?).

Para separar efectos se **estratifica**: se compara dentro de cada nivel de la
tercera variable.

### Paso 34. Interacción clase × género

In [ ]:
pd.read_parquet(GOLD / 'survival_by_class_sex.parquet').round(3)

### Paso 35. ¿El puerto importa, o es la clase?

La primera tabla mira el puerto **sin controlar nada**. La segunda estratifica
por clase. Si el efecto del puerto se desvanece dentro de cada clase, el puerto
no era la causa: era un indicador de la clase que embarcó allí.

In [ ]:
print("SIN estratificar — parece que el puerto importa:")
display(pd.read_parquet(GOLD / 'survival_by_embarked.parquet').round(3))

print("\nESTRATIFICADO por clase — ¿sigue importando?")
display(pd.read_parquet(GOLD / 'survival_by_embarked_class.parquet').round(3))

### Paso 36. ¿Pagar más ayudaba dentro de la misma clase?

Los cuartiles de tarifa se calculan **dentro de cada clase**, no sobre la
tarifa global: así se comparan pasajeros de la misma clase entre sí. `FareQ` 0
es el cuarto más barato de su clase, 3 el más caro.

In [ ]:
pd.read_parquet(GOLD / 'survival_by_fare_quartile_class.parquet').round(3)

### Paso 37. Tamaño de la familia

In [ ]:
pd.read_parquet(GOLD / 'survival_by_family.parquet').round(3)

In [ ]:
sns.pointplot(data=df, x='Familia', y='Survived')
plt.title('Supervivencia según número de familiares a bordo')
plt.xlabel('Familiares a bordo')
plt.ylabel('Tasa de supervivencia')
plt.show()

### Paso 38. Pasajeros atípicos

`IQR = Q3 − Q1`. Es atípico lo que queda fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

**Atípico no significa erróneo.** Una tarifa de 512 es real: alguien la pagó.
La regla señala candidatos a revisar, no filas a eliminar.

La detección de la edad usa `AgeOriginal`: sobre la imputada marcaría
pasajeros normales, porque el IQR encogido estrecha el umbral (H17).

In [ ]:
pd.read_parquet(GOLD / 'outlier_summary.parquet')

In [ ]:
# ¿Quiénes son los de tarifa más alta? Atípico no es lo mismo que erróneo
df.nlargest(10, 'Fare')[['Name', 'Pclass', 'Sex', 'Age', 'Fare', 'Familia', 'Survived']]

In [ ]:
# Una tasa sin su n no se puede interpretar: la celda Q×1ª son 2 pasajeros
print("Tasas:")
display(pd.read_parquet(GOLD / 'survival_by_embarked_class.parquet').round(3))
print("\nPasajeros detrás de cada tasa:")
display(pd.crosstab(df['Embarked'], df['Pclass']))